# Spatial Protein Clustering Analysis

This notebook identifies distinct spatial protein expression patterns in multiplexed microscopy images using KMeans clustering.

**Workflow:**
1. Load and preprocess images → binary protein masks
2. Global KMeans clustering → identify spatial compartments
3. Gene specificity analysis → assign proteins to compartments
4. Generate visualizations → color-coded spatial maps

---

## Setup

In [36]:
import matplotlib.pyplot as plt
import numpy as np
import pickle
import os
from skimage.filters import threshold_otsu, gaussian
from sklearn.cluster import KMeans
import pandas as pd
import glob
from tifffile import imread
from collections import defaultdict

## Configuration

In [19]:
# Analysis parameters
N_CLUSTERS = 10
CELL_LINE = 'A-431'
PATHWAY = 'organelle'
GEN_IMAGE_DIR = 'organelle_spatial_clustering_output' # Directory containing generated images

# Output directory
out_dir = f'spatial_clusters/{CELL_LINE}_{PATHWAY}_clustered_cell'
os.makedirs(out_dir, exist_ok=True)
os.makedirs('all_pixels', exist_ok=True)

# Color palette for visualization
STANDARD_COLORS = [
    (0, 0, 0), (255, 0, 0), (0, 0, 255), (0, 128, 0), (255, 255, 0), (255, 0, 255),
    (0, 255, 255), (255, 165, 0), (128, 0, 128), (0, 128, 128), (135, 206, 235), (128, 0, 0)
]

print(f"Output: {out_dir}")
print(f"Clusters: {N_CLUSTERS}")

Output: spatial_clusters/A-431_organelle_clustered_cell
Clusters: 10


## Helper Functions

In [4]:
def rgb_to_hex(rgb):
    """Convert RGB tuple to HEX color code."""
    return '#{:02x}{:02x}{:02x}'.format(int(rgb[0]), int(rgb[1]), int(rgb[2]))

def preprocess_image_stack(img_stack, thresh_multiplier=1.5, sigma=0.5):
    """Apply Gaussian smoothing and Otsu thresholding to image stack."""
    processed = img_stack.astype(np.float32)
    for i in range(processed.shape[2]):
        processed[:, :, i] = gaussian(processed[:, :, i], sigma=sigma)
        thresh = thresh_multiplier * threshold_otsu(processed[:, :, i])
        processed[:, :, i] = processed[:, :, i] > thresh
    return processed

def calculate_gene_specificity(expression_values, n_clusters):
    """Calculate specificity metrics for a gene across clusters."""
    total_expr = sum(expression_values)
    mean_expr = np.mean(expression_values)
    std_expr = np.std(expression_values)
    
    metrics = []
    for cluster_idx, expr in enumerate(expression_values):
        if expr == 0:
            continue
            
        # Calculate specificity ratio (vs other clusters)
        other_expr = total_expr - expr
        mean_other = other_expr / (n_clusters - 1) if n_clusters > 1 else 0
        specificity = expr / (mean_other + 1e-6)
        
        # Z-score
        z_score = (expr - mean_expr) / std_expr if std_expr > 0 else 0
        
        # Fold change
        fold_change = expr / (mean_expr + 1e-6)
        
        # Combined enrichment score
        enrichment = (2 * specificity + z_score + fold_change) / 4
        
        metrics.append({
            'cluster': cluster_idx,
            'expression': expr,
            'specificity': specificity,
            'z_score': z_score,
            'enrichment': enrichment,
            'fraction': expr / total_expr
        })
    
    return sorted(metrics, key=lambda x: x['enrichment'], reverse=True)

def assign_gene_to_clusters(metrics, mean_expr, max_clusters=3):
    """Assign gene to clusters based on specificity thresholds."""
    if not metrics:
        return []
    
    assigned = []
    top = metrics[0]
    
    # Check if top cluster meets thresholds
    if (top['z_score'] > 0.5 and top['specificity'] > 1.5 and top['fraction'] > 0.3):
        assigned.append(top['cluster'])
        
        # Add secondary clusters with relaxed criteria
        for m in metrics[1:]:
            if len(assigned) >= max_clusters:
                break
            if (m['enrichment'] > top['enrichment'] * 0.6 or
                m['specificity'] > 1.5 and m['expression'] > mean_expr * 1.5 or
                m['fraction'] > 0.3):
                assigned.append(m['cluster'])
    
    # Fallback: assign to most expressed if meaningful
    elif top['expression'] > mean_expr * 0.5:
        assigned.append(top['cluster'])
    
    return assigned

## Gene Selection

Select proteins spanning major cellular compartments.

In [ ]:
selected_gene_names = [
    # Nuclear envelope
    'TPR', 'LMNB1', 'SUN2', 'TMPO', 'LEMD2', 'TOR1AIP1', 'LBR', 'LMNB2', 'CASC3', 'NUP153',
    # Nucleolus
    'DDX47', 'RPF1', 'UTP6', 'NOL10', 'FTSJ3', 'UBTF', 'NAA50', 'NOP53', 'NOP56', 'NOLC1',
    'POLR2K', 'NOP16', 'UTP4', 'NOP2', 'MKI67', 'GNL3',
    # Nuclear speckles
    'TAF15', 'SMARCAD1', 'SRRM2', 'RBM25', 'PML', 'SMN2', 'RSL1D1', 'CENPC', 'RPS19',
    'RAN', 'HNRNPA1', 'H2AZ1', 'HNRNPC', 'HMGB1', 'HNRNPK', 'HNRNPA3', 'ATF4', 'MORF4L1',
    # Cytoskeleton
    'SEPTIN9', 'FGD4', 'ZYX', 'VCL', 'GSN', 'TUBA1A', 'CLIP1', 'PRC1', 'KIF18A', 'AURKB',
    'VIM', 'KRT19', 'KRT8', 'KRT17', 'DES', 'NES',
    # Mitochondria
    'CS', 'LRPPRC', 'SLC25A24', 'TIMM44', 'GCDH', 'TRAP1', 'MT-CYB', 'ATP5F1B', 'HSPD1',
    'COX4I1', 'ATP5ME', 'TOMM20', 'TOMM40', 'NDUFA13',
    # Golgi/ER
    'GOLGB1', 'GOLGA5', 'GALNT2', 'HSP90B1', 'CANX', 'PDIA3', 'CALR', 'P4HB',
    # Plasma membrane
    'STX4', 'SLC16A1', 'EZR', 'CTNNB1', 'MSN', 'CD9',
    # Endosomes
    'RAB5C', 'RAB7A', 'CD63', 'LAMP1'
]

selected_gene_names = list(set(selected_gene_names))
print(f"{len(selected_gene_names)} unique genes selected")

4 unique genes selected


## Prepare CSV for ProtVL Image Generation

Generate a CSV file that specifies which proteins to generate for clustering analysis.

In [12]:
# Read test images and parse metadata
input_image_dir = 'example_cell_reference_input'

image_paths = []
gene_list = []
cell_line_list = []

for filename in os.listdir(input_image_dir):
    # Parse filename: {cell_line}_{gene}_{...}.tiff
    parts = filename.strip('.tiff').split('_')
    cell_line = CELL_LINE

    
    # Filter for selected genes
    for gene in selected_gene_names:
        full_path = os.path.join(input_image_dir, filename)
        image_paths.append(full_path)
        gene_list.append(gene)
        cell_line_list.append(cell_line)

# Create CSV for ProtVL
generation_df = pd.DataFrame({
    'image_path': image_paths,
    'gene_name': gene_list,
    'cell_line_name': cell_line_list
})

generation_df.to_csv('organelle_spatial_clustering.csv', index=False)
print(f"Created CSV with {len(generation_df)} images for generation")
print(f"Cell lines: {generation_df['cell_line_name'].unique()}")
print(f"Genes: {len(generation_df['gene_name'].unique())} unique")

Created CSV with 60 images for generation
Cell lines: ['A-431']
Genes: 4 unique


## Generate Images with ProtVL

Before proceeding, run the ProtVL model to generate synthetic protein expression images. This step is performed outside the notebook.

**Command:**
```bash
python ordinary_sampler_standalone.py \
    --csv_file_path gen_model_test_images_II.csv \
    --model_path ./checkpoint-1020000/ \
    --vae_path ./vae \
    --cell_line_map_path cell_line_map.pkl \
    --antibody_map_path antibody_map.pkl \
    --mixed_precision no \
    --output_dir ./{GEN_IMAGE_DIR}\
    --batch_size 16 \
    --num_workers 0 \
    --num_inference_steps 100
```

**Outputs:**
- Generated images will be saved to `./{GEN_IMAGE_DIR}/`
- Filenames follow pattern: `{prefix}_{cell line}_{gene}_pred.tif`

**Note:** Update `GEN_IMAGE_DIR` in the next cell to point to your output directory.

## Load and Preprocess All Images

Apply Gaussian smoothing + Otsu thresholding to create binary protein masks.

In [32]:
# Get all prediction images
all_files = sorted(glob.glob(os.path.join(GEN_IMAGE_DIR, f'*_{CELL_LINE}_*_pred.tif')))

# Load all images (each is a different gene channel)
all_pixels = []
channels = []

for filepath in all_files:
    img = imread(filepath).astype(np.float32)
    channels.append(img)
    

# Stack into multi-channel image (H x W x C)
img_stack = np.stack(channels, axis=-1)

# Preprocess
img_stack = preprocess_image_stack(img_stack, thresh_multiplier=1.5)

# Collect pixels
pixels = img_stack.reshape(-1, img_stack.shape[2]).astype(np.int8)
all_pixels.append(pixels)

all_pixels = np.vstack(all_pixels)
print(f"\nTotal: {all_pixels.shape[0]:,} pixels from {len(channels)} gene channels")

np.save(f'all_pixels/{CELL_LINE}_{PATHWAY}_pixels.npy', all_pixels)


Total: 15,728,640 pixels from 60 gene channels


## Global KMeans Clustering

In [33]:
print("Performing KMeans clustering...")
kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=42)
kmeans.fit(all_pixels)

# Sort clusters by size
counts = np.bincount(kmeans.labels_)
sorted_indices = np.argsort(-counts)

# Reorder centers
reordered_centers = kmeans.cluster_centers_[sorted_indices]

print("\nCluster sizes:")
for i, idx in enumerate(sorted_indices):
    print(f"  Cluster {i}: {counts[idx]:,} ({100*counts[idx]/len(kmeans.labels_):.1f}%)")

del all_pixels  # Free memory

Performing KMeans clustering...


c:\Users\57539\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\cluster\_kmeans.py:1416: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)



Cluster sizes:
  Cluster 0: 13,699,839 (87.1%)
  Cluster 1: 1,387,551 (8.8%)
  Cluster 2: 115,781 (0.7%)
  Cluster 3: 101,882 (0.6%)
  Cluster 4: 96,265 (0.6%)
  Cluster 5: 81,630 (0.5%)
  Cluster 6: 80,606 (0.5%)
  Cluster 7: 79,226 (0.5%)
  Cluster 8: 48,413 (0.3%)
  Cluster 9: 37,447 (0.2%)


## Gene-Cluster Specificity Analysis

Identify which proteins are enriched in which spatial compartments.

In [34]:
gene_cluster_mapping = {}

for gene_idx, gene_name in enumerate(selected_gene_names):
    # Get expression across clusters
    expressions = [reordered_centers[c][gene_idx] for c in range(N_CLUSTERS)]
    
    if max(expressions) == 0:
        continue
    
    # Calculate metrics
    metrics = calculate_gene_specificity(expressions, N_CLUSTERS)
    
    # Assign to clusters
    assigned = assign_gene_to_clusters(metrics, np.mean(expressions))
    
    if assigned:
        gene_cluster_mapping[gene_name] = {
            'clusters': assigned,
            'metrics': metrics[:len(assigned)]
        }

print(f"Assigned {len(gene_cluster_mapping)} genes to clusters")

Assigned 4 genes to clusters


## Export Results

In [35]:
# Generate cluster colors
cluster_colors = [rgb_to_hex(STANDARD_COLORS[i]) for i in range(N_CLUSTERS)]

# Export gene-color mapping
with open(f'{out_dir}/gene_color_mapping.txt', 'w') as f:
    for gene in sorted(gene_cluster_mapping.keys()):
        for cluster_idx in gene_cluster_mapping[gene]['clusters']:
            f.write(f"{gene} {cluster_colors[cluster_idx]}\n")

# Export cluster analysis
with open(f'{out_dir}/cluster_analysis.txt', 'w') as f:
    f.write("CLUSTER ANALYSIS\n" + "="*80 + "\n\n")
    
    for i in range(N_CLUSTERS):
        # Find genes in this cluster
        cluster_genes = []
        for gene, data in gene_cluster_mapping.items():
            if i in data['clusters']:
                metric = next(m for m in data['metrics'] if m['cluster'] == i)
                cluster_genes.append((gene, metric['specificity'], metric['z_score']))
        
        cluster_genes.sort(key=lambda x: x[1], reverse=True)
        
        f.write(f"Cluster {i} ({cluster_colors[i]})\n")
        f.write(f"  Size: {counts[sorted_indices[i]]:,} pixels\n")
        f.write(f"  Genes: {len(cluster_genes)}\n")
        if cluster_genes:
            f.write(f"  Top genes: {', '.join([g[0] for g in cluster_genes[:5]])}\n")
        f.write("\n")

print(f"Results exported to {out_dir}/")

Results exported to spatial_clusters/A-431_organelle_clustered_cell/


## Generate Spatial Maps

Apply clustering to individual images and create color-coded visualizations.

In [42]:
print("Generating spatial maps...\n")

# Rebuild cell groups
all_files = sorted(glob.glob(os.path.join(GEN_IMAGE_DIR, f'*{CELL_LINE}*_pred.tif')))
cell_groups = defaultdict(list)

for filepath in all_files:
    filename = os.path.basename(filepath).replace('_pred.tif', '')
    
    gene_found = None
    for gene in selected_gene_names:
        if gene in filename:
            gene_found = gene
            break
    
    if gene_found:
        cell_prefix = filename.replace(f'_{gene_found}', '').replace(gene_found, '')
        cell_groups[cell_prefix].append((filepath, gene_found))

# Process each cell
for idx, (cell_prefix, gene_files) in enumerate(sorted(cell_groups.items())):
    # Sort by gene order
    gene_files_sorted = sorted(gene_files, key=lambda x: selected_gene_names.index(x[1]) 
                               if x[1] in selected_gene_names else 999)
    
    # Load channels
    channels = []
    for filepath, gene in gene_files_sorted:
        img = imread(filepath).astype(np.float32)
        channels.append(img[:, :, 1]) # channel 1 is the prediction
    
    # Stack and preprocess
    img_stack = np.stack(channels, axis=-1)
    img_stack = preprocess_image_stack(img_stack, thresh_multiplier=1.25)
    
    # Predict clusters
    h, w = img_stack.shape[:2]
    pixels = img_stack.reshape(-1, img_stack.shape[2]).astype(np.int8)
    labels = kmeans.predict(pixels)
    
    # Re-index to sorted order
    new_labels = np.zeros_like(labels)
    for i, sorted_idx in enumerate(sorted_indices):
        new_labels[labels == sorted_idx] = i
    
    # Create RGB image
    clustered = new_labels.reshape(h, w)
    rgb_img = np.zeros((h, w, 3), dtype=np.uint8)
    for i in range(N_CLUSTERS):
        rgb_img[clustered == i] = STANDARD_COLORS[i]
    
    # Save with cell prefix in filename
    safe_prefix = cell_prefix.replace('/', '_')
    plt.imsave(f'{out_dir}/{safe_prefix}_clustered.png', rgb_img)
    print(f"  Processed {idx + 1} cells", end='\r')
    
print(f"\n✓ Saved {len(cell_groups)} images to {out_dir}/")

Generating spatial maps...

  Processed 15 cells
✓ Saved 15 images to spatial_clusters/A-431_organelle_clustered_cell/


## Summary

**Outputs:**
- `gene_color_mapping.txt` - Gene-to-cluster assignments
- `cluster_analysis.txt` - Cluster statistics and top genes
- Color-coded spatial maps for each cell image

**Next steps:**
- Visualize cluster compositions
- Analyze co-localization patterns
- Compare across cell lines or conditions